# Notebook 31: Classifying ImageNet in VAE Latent Space

---

## What This Notebook Covers

This is the course's final notebook, and it makes a point that is easy to miss: a VAE's latent space is not only good for *generation* — it is an excellent, compact, pretrained **feature representation for discriminative tasks too**. Here we take the same Stable Diffusion VAE from the latent-diffusion notebook, encode all of **ImageNet** into 4×32×32 latents, and train a **wide ResNet classifier** to predict the 1000 ImageNet classes *from the latents alone*. We will learn:

1. **Discriminative use of a generative model's latents** — classify on VAE codes instead of pixels, the "encode once, train a cheap head" transfer paradigm
2. **Precomputing latents to per-image `.npy` files** — a label-preserving storage layout (folder = class), contrasting with the single memmap of notebook 30
3. **Normalizing latents** — the per-channel mean/std of the SD-VAE latent space, and standardizing to ~unit statistics for the classifier
4. **Data augmentation *in latent space*** — applying `Pad → RandomCrop → RandErase` to the 4×32×32 latent, not the image
5. **A "wide-ish" ResNet** — the classifier architecture (`ResBlock`, `res_blocks`, `get_dropmodel`) reading 4-channel latents → 1000 logits
6. **Training** a strong ImageNet classifier that never touches a pixel after the one-time encode

> **A course-arc correction, up front.** You might expect notebook 31 to be *conditional generation* (class-conditioned latent diffusion — the natural sequel to notebooks 28–30). It is not. It is **classification**: the *discriminative* mirror of the latent-diffusion story. Notebook 30 showed that VAE latents are a good space to *generate* in; notebook 31 shows the same latents are a good space to *recognize* in. Same encoder, opposite task. I flag this so the framing lands correctly.

---

## Why Classify on Latents?

By now the generative story is familiar: encode an image to a small VAE latent, diffuse there, decode. This notebook asks a different question — **what else is that latent good for?** — and answers: *a lot*, including plain classification.

Training an ImageNet classifier directly on 3×256×256 pixels is expensive. But the SD VAE has already done the hard perceptual work: it compresses each image ~48× into a 4×32×32 latent that preserves everything perceptually important. So instead of a classifier that must *both* learn low-level vision *and* the class boundaries, we:

1. **Encode every image once** with the frozen VAE → a tiny latent (the expensive step, done a single time).
2. **Train a compact ResNet on the latents** → 48× fewer input numbers, so training is fast and the model can be modest.

This is exactly the **foundation-model transfer paradigm**: take a large model pretrained on a huge corpus, freeze it, extract its representation once for your data, and train a small task-specific head on top of those embeddings. The VAE here plays the role of the frozen foundation model; the ResNet is the downstream head; the `.npy` latents are the precomputed embeddings.

**Climate / foundation-model bridges (squarely on your career path).**
- **This *is* the foundation-model workflow you want to do for climate.** Swap the SD VAE for a pretrained climate/weather foundation model (an encoder trained on ERA5, satellite stacks, etc.), encode your dataset once into embeddings, and train small downstream heads — for classification (land cover, regime identification), regression (σ_LST, downscaled fields), or detection. The notebook is a clean, end-to-end template of that pattern.
- **Feature-space augmentation ↔ embedding-space augmentation** used in representation learning (mixup/manifold-mixup, feature dropout). Augmenting the latent (not the image) is the same idea you would use when your inputs *are* embeddings.
- **Precompute-embeddings-to-disk ↔ your HPC preprocessing**, as in notebook 30 — but here stored per-example with labels encoded in the directory tree.

---

## Prerequisites

You should be comfortable with:

- **VAE latents** (notebooks 29–30) — the pretrained SD `AutoencoderKL`, `latent_dist.mean`, the 4×32×32 latent, decoding back to images.
- **ResNets** (notebooks 13, 24) — residual blocks, the `convs(x) + idconv(x)` structure, wide ResNets. The classifier here is a recap applied to latent inputs.
- **Data augmentation** (notebook 14) — `RandomCrop`, random erasing, and why augmentation regularizes.
- **The `miniai` training loop** — `Learner`, `MetricsCB`, `MulticlassAccuracy`, `OneCycleLR`, `MixedPrecision`, `GeneralRelu`, `init_weights`.
- **Normalization / standardization** — subtract mean, divide by std, per channel.

---


## Google Colab Setup

Run the cells below **once** at the start of each Colab session. They mount Google Drive, set the working directory, install required packages, and clone the `miniai` library from the [fast.ai Part 2 course repo](https://github.com/fastai/course22p2).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.chdir('/content/drive/MyDrive/Fast.AI_Colab')
print(os.getcwd())

In [ ]:
# Install required packages
!pip install -q fastcore fastai diffusers datasets torcheval accelerate einops timm

# Clone the fast.ai Part 2 course repo to get the miniai library
if not os.path.exists('course22p2'):
    !git clone https://github.com/fastai/course22p2.git

# Add miniai to the Python path
import sys
sys.path.insert(0, os.path.join(os.getcwd(), 'course22p2'))

# Verify miniai is accessible
try:
    import miniai
    print(f'miniai loaded successfully from: {miniai.__file__}')
except ImportError:
    print('ERROR: miniai not found. Check that course22p2 was cloned correctly.')

---

*The cells above are Colab-specific setup. The content below is the same as the source notebook (`31_imgnet_latents-widish_explained.ipynb`). The `CUDA_VISIBLE_DEVICES` cell already targets index `0` (correct for Colab's single GPU), so it was left unchanged.*

> **Big heads-up before running (this notebook is heavy and needs full ImageNet).**
> - **Dataset.** It expects the **full ILSVRC/ImageNet** dataset laid out at `data/ILSVRC/Data/CLS-LOC/{train,val}/<synset>/*.JPEG`, plus `imagenet_lsvrc_2015_synsets.txt` and `words.txt`. ImageNet is **not** downloadable from a public URL like the LSUN archive &mdash; you must obtain it separately (Kaggle's ImageNet Object Localization Challenge, or an academic mirror) and place it under `data/ILSVRC/`. Point `path_data` at a Drive folder so it persists across sessions.
> - **Encoding.** Encoding all ~1.3M ImageNet images through the VAE and writing per-image `.npy` files is a very long, storage-heavy pass (hundreds of GB of latents). This will not finish in a free Colab session &mdash; run it on a persistent machine, or work with a subset of synsets.
> - **Training.** 40 epochs over the full latent set is a multi-hour+ job. For a smoke test, restrict to a handful of classes and a few epochs.
> - **`torch.save(learn.model, 'models/imgnet-latents')`** writes to `models/` &mdash; create that folder (or point it at Drive) first.
>
> In short: this final notebook is best read for its *ideas* (latent-space classification, foundation-model transfer) unless you have ImageNet staged and real compute. The mechanics are identical to the smaller notebooks you can run end-to-end.

---

# Part 1: Setup and the Pretrained VAE

Imports, environment tweaks, and loading the frozen Stable Diffusion VAE that will produce our latents. As in notebook 30, the VAE is used only for inference (encode, and occasionally decode for visualization).

---


In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES']='0'
os.environ['OMP_NUM_THREADS']='1'

**What does the code above do?**

Pins to GPU 0 and caps OpenMP threads to 1. `OMP_NUM_THREADS=1` avoids oversubscription when many `DataLoader` workers each spin up their own thread pools — a common throughput fix for data-heavy pipelines like encoding all of ImageNet.


In [ ]:
import pickle,gzip

from glob import glob
from torcheval.metrics import MulticlassAccuracy

from miniai.imports import *

**What does the code above do?**

Imports for file-list caching (`pickle`, `gzip`), globbing, the classification metric (`MulticlassAccuracy`), and the full `miniai` toolkit. Note this notebook imports `miniai.imports` but **not** `miniai.diffusion` — there is no diffusion here; the ResNet and training pieces come from the core library.


In [ ]:
from fastprogress import progress_bar
from diffusers import AutoencoderKL

**What does the code above do?**

`progress_bar` for the latent-precompute loop, and `AutoencoderKL` — the Stable Diffusion VAE — for encoding. (You may see a harmless Triton warning on import, as in notebook 30.)


In [ ]:
torch.set_printoptions(precision=5, linewidth=140, sci_mode=False)
torch.manual_seed(1)
mpl.rcParams['figure.dpi'] = 70

set_seed(42)
if fc.defaults.cpus>8: fc.defaults.cpus=8

**What does the code above do?**

Standard reproducibility/display boilerplate, and the 8-worker cap for data loading.


In [ ]:
path_data = Path('data')/'ILSVRC'
path = path_data/'Data'/'CLS-LOC'

dest = path_data/'latents'
dest.mkdir(exist_ok=True)

**What does the code above do?**

Sets up the ImageNet paths. `path` points at the raw ILSVRC images (the standard `Data/CLS-LOC` layout with `train/`, `val/` and per-class subfolders). `dest = data/ILSVRC/latents` is where the precomputed latent `.npy` files will be written, mirroring the image directory tree.


In [ ]:
vae = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-ema").cuda().requires_grad_(False)

**What does the code above do?**

Loads the same pretrained SD VAE as notebook 30 and freezes it (`requires_grad_(False)`). It maps `3×H×W` images to `4×(H/8)×(W/8)` latents. Frozen because we only use it to *produce* the latents — the classifier trains on the latents, never on the VAE.


---

# Part 2: ImageNet Images

We wrap the ImageNet files in a dataset that reads, center-crops to a square, and resizes to 256×256 — the size the VAE expects. A small but nice engineering touch: the (slow) file-list glob is cached to disk so subsequent runs start instantly.

---


In [ ]:
class ImagesDS:
    def __init__(self, path, spec):
        cache = path/'files.zpkl'
        if cache.exists():
            with gzip.open(cache) as f: self.files = pickle.load(f)
        else:
            self.files = glob(str(path/spec), recursive=True)
            with gzip.open(cache, 'wb', compresslevel=1) as f: pickle.dump(self.files, f)

    def __len__(self): return len(self.files)

    def __getitem__(self, i):
        f = self.files[i]
        im = read_image(f, mode=ImageReadMode.RGB)/255
        im = TF.resize(TF.center_crop(im, min(im.shape[1:])), 256)
        return im,f

**What does the code above do?**

An image dataset with a cached file list. In `__init__`, globbing millions of ImageNet paths is slow, so the first run pickles the file list to a gzipped cache (`files.zpkl`) and every later run loads it in a blink. `__getitem__` reads an image, **center-crops to a square** (`center_crop(im, min(H,W))` takes the largest centered square), and **resizes to 256×256**. It returns `(im, f)` — the image *and its file path* `f`, because the path encodes the class label (the parent folder is the synset) and the output location for its latent.


In [ ]:
ds = ImagesDS(path, '**/*.JPEG')
dl = DataLoader(ds, batch_size=64, num_workers=fc.defaults.cpus)

**What does the code above do?**

Instantiates the dataset over all ImageNet `.JPEG` files and wraps it in a `DataLoader` (batch size 64) for the one-time encoding pass.


In [ ]:
xb,yb = next(iter(dl))
xe = vae.encode(xb.cuda())
xs = xe.latent_dist.mean
xs.shape

torch.Size([64, 4, 32, 32])

**What does the code above do?**

Encodes one batch and inspects the latent shape: **`(64, 4, 32, 32)`** — 64 images, each compressed from 3×256×256 to a 4×32×32 latent (the same 48× reduction as notebook 30). `xs = xe.latent_dist.mean` is the deterministic latent (the distribution's mean, i.e. the `mu` head). These 4-channel latents are what the classifier will consume.


## Deep Dive: Classifying in Latent Space

We are about to build a classifier that never sees a pixel — it reads 4×32×32 VAE latents and outputs ImageNet class logits. Why is that a good idea, and what does it teach?

### The representation is already good

A classifier normally has to learn *two* things at once: (1) low-level and mid-level vision — edges, textures, parts — and (2) the mapping from those features to class labels. Learning (1) from raw pixels is most of the work, and it needs a big network and a lot of compute.

But the SD VAE was trained (on a huge image corpus) to compress images into latents that **retain everything needed to reconstruct them** — which necessarily means the latent already encodes rich perceptual structure. So much of job (1) is *already done* inside the latent. A classifier starting from latents can spend its capacity on job (2), the actual class boundaries, and can be smaller and train faster.

### The numbers

- **Input size:** 4×32×32 = 4,096 numbers per image, vs 3×256×256 = 196,608 for pixels — **48× smaller**. The classifier's first layers, and its whole memory/compute budget, shrink accordingly.
- **One-time encode:** the expensive VAE pass runs once over the dataset (Part 3); training then reads cheap `.npy` latents. (Same economics as the latent-diffusion precompute.)

### This is the foundation-model transfer pattern

Step back and the structure is completely general:

| Role here | General foundation-model transfer |
|-----------|-----------------------------------|
| SD VAE (frozen) | A large model pretrained on a huge corpus |
| `latent_dist.mean` | The extracted embedding / representation |
| `.npy` files on disk | Precomputed embeddings for your dataset |
| The ResNet | A small, task-specific downstream head |
| ImageNet classification | Your downstream task |

Freeze the big model, extract its representation once, train a light head on the embeddings. That is how most practical transfer learning works today — and it is precisely the workflow you would use to build a downstream model on top of a **climate/weather foundation model's** embeddings. This notebook is a minimal, end-to-end instance of it, which is why it is a fitting capstone.

> **A caveat worth naming.** The VAE was trained for *reconstruction*, not *classification*, so its latent is not perfectly tuned for discriminability — a purpose-built self-supervised encoder (e.g. a contrastive or masked-autoencoder model) would give a representation more aligned to recognition. That the VAE latent classifies ImageNet *well anyway* is exactly the point: a good generative representation is broadly useful, discriminatively too.


In [ ]:
show_images(((xs[:16,:3])/4).sigmoid(), imsize=2)

**What does the code above do?**

Visualizes the raw latents (first 3 of 4 channels, scaled by `/4` and `sigmoid`-squashed for display, exactly as in notebook 30). **What you should see:** 16 small, abstract, colorful 32×32 blobs — the compressed representations, not literal thumbnails. (Image omitted.)


In [ ]:
xd = to_cpu(vae.decode(xs))
show_images(xd['sample'][:16].clamp(0,1), imsize=2)

**What does the code above do?**

Decodes the latents back to images as a sanity check. **What you should see:** 16 recognizable ImageNet reconstructions — confirming the latents faithfully represent the images (and therefore carry the information a classifier needs). (Image omitted.)


---

# Part 3: Precomputing Latents to Per-Image `.npy` Files

As in notebook 30, we encode the whole dataset once and store the latents on disk. But here we use a different layout: **one `.npy` file per image, mirroring the class-folder directory tree** — so the folder structure preserves the labels, and standard folder-based dataset tooling can read them back.

---


In [ ]:
if not dest.exists():
    dest.mkdir()
    for xb,yb in progress_bar(dl):
        eb = to_cpu(vae.encode(xb.cuda()).latent_dist.mean).numpy()
        for ebi,ybi in zip(eb,yb):
            ybi = dest/Path(ybi).relative_to(path).with_suffix('')
            (ybi.parent).mkdir(parents=True, exist_ok=True)
            np.save(ybi, ebi)

**What does the code above do?**

The encode-once pass. For each image batch it computes the latent means and then, **per image**, writes a `.npy` file at the mirrored path under `dest`:

- `Path(ybi).relative_to(path)` strips the image root, giving e.g. `train/n01440764/n01440764_18.JPEG`.
- `.with_suffix('')` drops the extension; `np.save` re-adds `.npy`.
- `mkdir(parents=True, exist_ok=True)` recreates the class subfolders under `latents/`.

So the result is `latents/train/n01440764/n01440764_18.npy`, one small array per image, **class encoded in the folder name** (`n01440764` is the synset). This runs once (guarded by `if not dest.exists()`).

**Why per-`.npy` instead of one big memmap (as in notebook 30)?** Two reasons: (1) **labels come for free** — the ImageNet class is the parent directory, so no separate label array is needed; (2) it plugs directly into ordinary folder-based dataset code and standard train/val splits. The trade-off is millions of small files (slower to enumerate — hence the pickled file-list cache in Part 2). The memmap wins for a single unlabeled stream; the per-file tree wins when you need labels and folder semantics.


In [ ]:
class NumpyDS(ImagesDS):
    def __getitem__(self, i):
        f = self.files[i]
        im = np.load(f)
        return im,f

**What does the code above do?**

A dataset for reading the precomputed latents. It **subclasses `ImagesDS`** (reusing its cached-file-list `__init__`) and overrides only `__getitem__` to `np.load` a latent array instead of reading and processing an image. Returns `(latent, filepath)` — the path again carries the label. This is what training reads from; the VAE is out of the loop entirely from here on.


In [ ]:
bs = 128

**What does the code above do?**

Training batch size 128 — larger than the encoding pass's 64, affordable because latents are tiny.


In [ ]:
tds = NumpyDS(dest/'train', '**/*.npy')
vds = NumpyDS(dest/'val', '**/*.npy')

**What does the code above do?**

Builds the train and validation latent datasets by pointing `NumpyDS` at the `train/` and `val/` subtrees of the latents directory. The ImageNet-provided train/val split is respected (it is baked into the folder structure).


---

# Part 4: Normalizing the Latents

Before training, we standardize the latents to roughly zero mean and unit variance per channel. This deep dive explains the specific statistics and why standardization matters here (and how to undo it for visualization).

---


In [ ]:
tdl = DataLoader(tds, batch_size=bs, num_workers=0)
xb,yb = next(iter(tdl))
xb.mean((0,2,3)),xb.std((0,2,3))

(tensor([ 5.23983,  2.59586,  0.45112, -2.28669]),
 tensor([3.94172, 4.42124, 3.24268, 3.09760]))

**What does the code above do?**

Measures the **per-channel statistics** of the latents from one batch: `mean((0,2,3))` and `std((0,2,3))` average over batch and spatial dims, leaving one number per latent channel. The output shows the SD-VAE latents are **not** zero-mean/unit-variance: means range from ≈ +5.2 to ≈ −2.3, stds ≈ 3–4. Each of the 4 channels has its own scale and offset — exactly what standardization needs to fix.


In [ ]:
xmean,xstd = (tensor([ 5.37007,  2.65468,  0.44876, -2.39154]),
 tensor([3.99512, 4.44317, 3.21629, 3.10339]))

**What does the code above do?**

Hard-codes the per-channel mean and std (measured over a larger sample than the single batch above, hence slightly different values). Freezing these as constants makes normalization deterministic and reproducible across runs — the classifier always sees latents standardized the same way.


## Deep Dive: Latent Normalization (and `denorm`)

### Why standardize at all

Neural networks train best when their inputs are roughly zero-mean and unit-variance per feature. If one input channel has mean +5 and std 4 while another has mean −2 and std 3 (as our latents do), the first layer's weights must learn to compensate for those offsets and scales before doing anything useful — wasted capacity and a harder optimization landscape. **Standardizing** each channel,

$$\hat{x}_c = \frac{x_c - \mu_c}{\sigma_c},$$

puts all four latent channels on the same footing: zero-centered, unit-scaled. Then BatchNorm and the learned weights can focus on the signal, not the scale. This is the same per-channel image normalization you always apply (ImageNet RGB means/stds) — just measured on the *latent* channels instead of RGB.

### The specific numbers tell a story

The measured stats (`mean ≈ [5.4, 2.7, 0.4, −2.4]`, `std ≈ [4.0, 4.4, 3.2, 3.1]`) are a property of *the SD VAE's latent space*, not of ImageNet specifically — any images encoded by this VAE land in a latent distribution with roughly these per-channel offsets and a std around 3–4. (Recall notebook 30 used a crude single scalar `×0.2 ≈ 1/5` to bring latents near unit variance for diffusion; here we do the proper thing — a **per-channel** mean subtraction and std division — because a classifier benefits from precise standardization.)

### `denorm` — undoing it to decode

```python
def denorm(x): return (x*xstd[:,None,None]+xmean[:,None,None])
```

Because the classifier works on *normalized* latents, any time we want to **decode a normalized latent back to an image** (to visualize what the model is looking at), we must first invert the standardization: multiply by `xstd` and add `xmean` back, per channel. The `[:,None,None]` reshapes the length-4 stat vectors to `(4,1,1)` so they broadcast across the 32×32 spatial dims. `denorm` is the exact inverse of `norm_tfm` below — normalize to feed the classifier, denormalize to feed the VAE decoder.


---

# Part 5: Transforms, Augmentation, and Labels

We assemble the input pipeline: a normalization transform, **augmentation applied in latent space**, and the label lookup that turns a file path into a class id. A small `TfmDS` wrapper applies per-item transforms.

---


In [ ]:
class TfmDS:
    def __init__(self, ds, tfmx=fc.noop, tfmy=fc.noop): self.ds,self.tfmx,self.tfmy = ds,tfmx,tfmy
    def __len__(self): return len(self.ds)
    def __getitem__(self, i):
        x,y = self.ds[i]
        return self.tfmx(x),self.tfmy(y)

**What does the code above do?**

A thin dataset wrapper that applies a transform to the input (`tfmx`) and a transform to the label (`tfmy`) on the fly. It lets us keep the raw `NumpyDS` clean and layer normalization/augmentation (for `x`) and path→class-id conversion (for `y`) on top, differently for train vs validation.


In [ ]:
id2str = (path_data/'imagenet_lsvrc_2015_synsets.txt').read_text().splitlines()
str2id = {v:k for k,v in enumerate(id2str)}

**What does the code above do?**

Loads the ImageNet **synset** list — 1000 lines like `n01440764`, in the canonical class order. `id2str[k]` maps a class index → synset string; `str2id` is the inverse (synset → index). Since each latent's folder name *is* its synset, `str2id[folder_name]` gives the integer class label the classifier trains against.


In [ ]:
aug_tfms = nn.Sequential(T.Pad(2), T.RandomCrop(32), RandErase())
norm_tfm = T.Normalize(xmean, xstd)

**What does the code above do?**

Two transform pipelines:

- **`norm_tfm = T.Normalize(xmean, xstd)`** — the per-channel standardization from the deep dive, applied to every latent (train and val).
- **`aug_tfms`** — data augmentation applied to *training* latents only: `Pad(2)` grows the 32×32 latent to 36×36, `RandomCrop(32)` takes a random 32×32 window (a random small translation), and `RandErase()` blanks a random rectangle (random erasing, notebook 14). Crucially, **these augmentations operate on the latent, not the image** — the subject of the next deep dive.


## Deep Dive: Data Augmentation in Latent Space

Augmentation normally acts on *pixels*: crop, flip, jitter the image, then encode/feed it. Here we do something less common — we augment the **latent** directly (`Pad → RandomCrop → RandErase` on the 4×32×32 code). Why does this work, and what are the caveats?

### Why it works at all

The VAE is (approximately) **spatially structured**: because it downsamples convolutionally by 8×, a latent's `(row, col)` position corresponds to a patch of the image, and neighboring latent positions correspond to neighboring image patches. So geometric operations on the latent grid have coherent image-space meaning:

- **`Pad(2) → RandomCrop(32)`** shifts the latent by a few positions — a small **translation** of the underlying image (each latent step ≈ 8 pixels). This is the classic translation augmentation, done cheaply on the small latent instead of the big image.
- **`RandErase`** blanks a latent rectangle — roughly erasing a *region* of the image, the latent-space version of random erasing / cutout, which forces the classifier not to rely on any single region.

### Why do it in latent space (not on pixels)

- **It is free at training time.** The latents are already computed and tiny; augmenting them is a couple of cheap tensor ops. Augmenting pixels would require either storing full images (defeating the 48× compression) or re-encoding through the VAE every step (expensive).
- **It matches the input the model actually sees.** The classifier consumes latents, so regularizing in latent space directly perturbs its true input distribution.

### The caveats (be honest)

- **Not all image augmentations have a clean latent analog.** A horizontal *flip* of the image is **not** simply a flip of the latent grid — the VAE's convolutions are not flip-equivariant, so flipping the latent produces something that does not decode to a flipped image. Translation and erasing survive the latent because they respect the approximate spatial structure; color jitter, flips, and rotations generally do not. That is why the pipeline here uses only pad/crop/erase.
- **Latent augmentation is approximate.** Even translation is only *approximately* meaningful, because the 8× downsampling and the VAE's receptive field blur the exact pixel-to-latent correspondence. In practice it regularizes well enough to help; just know it is a convenient approximation, not an exact image-space augmentation.

**Bridge to representation learning.** Perturbing a learned embedding to regularize a downstream head is a general technique (feature-space augmentation, manifold mixup, embedding dropout). When your model's inputs *are* embeddings — as in foundation-model transfer — augmenting in that embedding space, mindful of which perturbations preserve meaning, is exactly this move.


In [ ]:
def tfmx(x, aug=False):
    x = norm_tfm(tensor(x))
    if aug: x = aug_tfms(x[None])[0]
    return x

def tfmy(y): return tensor(str2id[Path(y).parent.name])

tfm_tds = TfmDS(tds, partial(tfmx, aug=True), tfmy)
tfm_vds = TfmDS(vds, tfmx, tfmy)

**What does the code above do?**

Wires up the transforms:

- **`tfmx`** normalizes the latent, then (if `aug=True`) applies the latent augmentations. The `x[None]`/`[0]` adds and removes a batch axis because the `T.*` transforms expect a batched tensor.
- **`tfmy`** turns a file path into a class id: `Path(y).parent.name` is the synset folder (e.g. `n01440764`), and `str2id[...]` maps it to the integer label.
- `tfm_tds` uses **`aug=True`** (augment training latents); `tfm_vds` uses the default `aug=False` (validation is normalized but not augmented — you never augment the eval set).


In [ ]:
def denorm(x): return (x*xstd[:,None,None]+xmean[:,None,None])

**What does the code above do?**

The inverse of `norm_tfm` (per the normalization deep dive): rescale a normalized latent back to the VAE's native latent range so it can be decoded to an image. Used just below to visualize a training batch.


In [ ]:
dls = DataLoaders(*get_dls(tfm_tds, tfm_vds, bs=bs, num_workers=8))

**What does the code above do?**

Builds the train/valid `DataLoaders` over the transformed latent datasets, batch size 128, 8 workers. Each batch yields `(normalized_augmented_latent, class_id)` — the classifier's training data.


In [ ]:
all_synsets = [o.split('\t') for o in (path_data/'words.txt').read_text().splitlines()]
synsets = {k:v.split(',', maxsplit=1)[0] for k,v in all_synsets if k in id2str}

**What does the code above do?**

Builds human-readable class names. `words.txt` maps each synset id to its WordNet gloss (tab-separated, e.g. `n01440764\ttench, Tinca tinca`). This keeps only the 1000 synsets in our label set and takes the **first** name before the comma (`tench`), giving a `synset → readable_name` dict for titling visualizations.


In [ ]:
xb,yb = next(iter(dls.train))
titles = [synsets[id2str[o]] for o in yb]
xb.mean(),xb.std()

(tensor(-0.02974), tensor(0.97078))

**What does the code above do?**

Grabs a training batch, builds readable `titles` (class id → synset → name), and checks the batch statistics: **mean ≈ 0, std ≈ 0.97** — confirming normalization worked (the latents are now standardized to ~zero-mean/unit-variance, ready for the network). Compare to the raw per-channel means of +5.2…−2.3 before normalization.


In [ ]:
xd = to_cpu(vae.decode(denorm(xb[:9]).cuda()))
show_images(xd['sample'].clamp(0,1), imsize=4, titles=titles[:9])

**What does the code above do?**

Sanity-checks the full pipeline visually: take 9 training latents, **`denorm`** them back to native latent range, **decode** through the VAE to images, and show them titled with their class names. **What you should see:** 9 recognizable ImageNet images, each captioned with its class (and possibly showing the pad/crop/erase augmentation — a slight shift or an erased patch). This confirms latents, labels, normalization, and augmentation are all correctly aligned before training. (Image omitted.)


---

# Part 6: The Wide-ish ResNet Classifier

Now the model: a ResNet that reads 4-channel latents and outputs 1000 class logits. The residual-block machinery is the notebook-13/24 ResNet, recapped here and adapted to a 4-channel input. "Widish" refers to the generous channel widths.

---


In [ ]:
act_gr = partial(GeneralRelu, leak=0.1, sub=0.4)
iw = partial(init_weights, leaky=0.1)

opt_func = partial(optim.AdamW, eps=1e-5)
metrics = MetricsCB(accuracy=MulticlassAccuracy())
cbs = [DeviceCB(), metrics, ProgressCB(plot=True), MixedPrecision()]

**What does the code above do?**

Training ingredients:

- **`act_gr`** — `GeneralRelu` with `leak=0.1` (leaky-ReLU-style negative slope) and `sub=0.4` (subtracts 0.4 to re-center the post-activation mean toward 0, notebook 11's trick). The activation used throughout the ResNet.
- **`iw`** — Kaiming init matched to that leak.
- **`opt_func`** — AdamW.
- **`metrics`** — tracks `MulticlassAccuracy` (this is classification, so accuracy is the headline metric alongside the cross-entropy loss).
- **`cbs`** — device placement, metrics, live plot, mixed precision.


In [ ]:
def conv(ni, nf, ks=3, stride=1, act=nn.ReLU, norm=None, bias=True):
    layers = []
    if norm: layers.append(norm(ni))
    if act : layers.append(act())
    layers.append(nn.Conv2d(ni, nf, stride=stride, kernel_size=ks, padding=ks//2, bias=bias))
    return nn.Sequential(*layers)

def _conv_block(ni, nf, stride, act=act_gr, norm=None, ks=3):
    return nn.Sequential(conv(ni, nf, stride=1     , act=act, norm=norm, ks=ks),
                         conv(nf, nf, stride=stride, act=act, norm=norm, ks=ks))

class ResBlock(nn.Module):
    def __init__(self, ni, nf, stride=1, ks=3, act=act_gr, norm=None):
        super().__init__()
        self.convs = _conv_block(ni, nf, stride, act=act, ks=ks, norm=norm)
        self.idconv = fc.noop if ni==nf else conv(ni, nf, ks=1, stride=1, act=None, norm=norm)
        self.pool = fc.noop if stride==1 else nn.AvgPool2d(2, ceil_mode=True)

    def forward(self, x): return self.convs(x) + self.idconv(self.pool(x))

def res_blocks(n_bk, ni, nf, stride=1, ks=3, act=act_gr, norm=None):
    return nn.Sequential(*[
        ResBlock(ni if i==0 else nf, nf, stride=stride if i==n_bk-1 else 1, ks=ks, act=act, norm=norm)
        for i in range(n_bk)])

def get_dropmodel(nfs, nbks, act=act_gr, norm=nn.BatchNorm2d, drop=0.2):
    layers = [nn.Conv2d(4, nfs[0], 5, padding=2)]
    layers += [res_blocks(nbks[i], nfs[i], nfs[i+1], act=act, norm=norm, stride=2)
               for i in range(len(nfs)-1)]
    layers += [act_gr(), norm(nfs[-1]), nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(drop)]
    layers += [nn.Linear(nfs[-1], 1000, bias=False), nn.BatchNorm1d(1000)]
    return nn.Sequential(*layers).apply(iw)

**What does the code above do?**

The classifier architecture — the notebook-13/24 pre-activation ResNet, adapted for latent inputs. Piece by piece:

| Component | Role |
|-----------|------|
| `conv` | `norm → act → Conv2d` (pre-activation conv) |
| `_conv_block` | two convs; the second carries the stride (downsample) |
| `ResBlock` | `convs(x) + idconv(pool(x))` — the residual: main path plus a shortcut that `AvgPool`s (if stride>1) and 1×1-projects (if channels change) so the add lines up |
| `res_blocks` | a stack of `n_bk` ResBlocks; only the **last** one strides (downsamples), the rest keep resolution |
| `get_dropmodel` | assembles the whole net |

Read `get_dropmodel`:

- **`nn.Conv2d(4, nfs[0], 5, padding=2)`** — the stem takes **4 channels** (the latent depth, not RGB) and lifts to `nfs[0]` with a 5×5 conv.
- A sequence of `res_blocks` stages, each **striding by 2** (halving the 32×32 latent through the network) and widening channels per `nfs`.
- A head: activation, norm, **global average pool**, flatten, **dropout**, a `Linear(nfs[-1], 1000)`, and a final `BatchNorm1d(1000)` on the logits.
- `.apply(iw)` initializes all layers.

With `nfs=(32,64,128,512,1024)` and `nbks=(1,2,4,3)` (used next), it is a genuinely capable ImageNet classifier — but a *small, fast* one, because its input is a 4×32×32 latent, not a 3×256×256 image. That size reduction is the whole payoff of working in latent space.


---

# Part 7: Training and Saving

Standard classification training: cross-entropy, AdamW, one-cycle LR, 40 epochs, accuracy tracked.

---


In [ ]:
epochs = 40
lr = 1e-2
tmax = epochs * len(dls.train)
sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)
xtra = [BatchSchedCB(sched)]
model = get_dropmodel(nbks=(1,2,4,3), nfs=(32, 64, 128, 512, 1024), drop=0.1)
learn = Learner(model, dls, F.cross_entropy, lr=lr, cbs=cbs+xtra, opt_func=opt_func)

**What does the code above do?**

Assembles the training run:

| Component | Choice |
|-----------|--------|
| Model | `get_dropmodel(nbks=(1,2,4,3), nfs=(32,64,128,512,1024), drop=0.1)` — a 4-stage wide ResNet, ~10 residual blocks, on 4-channel latents |
| Loss | `F.cross_entropy` — standard classification loss |
| Optimizer | AdamW, `eps=1e-5` |
| Schedule | OneCycle, `max_lr=1e-2`, over all 40 epochs |
| Callbacks | device, accuracy metrics, live plot, mixed precision, per-batch LR schedule |

`nbks=(1,2,4,3)` sets the number of residual blocks per stage (deeper in the middle), and `nfs` the channel widths (up to 1024) — the "widish" configuration. `drop=0.1` is light dropout before the classifier head for regularization.


In [ ]:
learn.fit(epochs)

**What does the code above do?**

Trains for 40 epochs. **What you should see:** cross-entropy loss falling and **top-1 accuracy climbing** on the ImageNet validation set. Training is comparatively fast for ImageNet because every input is a 4×32×32 latent rather than a full image — the latent-space payoff again. (Plot/metrics omitted.)


In [ ]:
torch.save(learn.model, 'models/imgnet-latents')

**What does the code above do?**

Saves the trained classifier to `models/imgnet-latents`. Note it saves the **whole model** object (not just `state_dict`), so reloading needs the class definitions in scope. This is the trained downstream head — a classifier that recognizes ImageNet from VAE latents.


---

# Summary and What's Next

### What we built

| Step | What | Why |
|------|------|-----|
| Encode | SD VAE → 4×32×32 latents for all of ImageNet | 48× smaller, pretrained perceptual features |
| Store | one `.npy` per image, folder = class | labels come free from the directory tree |
| Normalize | per-channel `(x − xmean)/xstd` | standardize the VAE's non-centered latent channels |
| Augment | `Pad → RandomCrop → RandErase` **on latents** | cheap, latent-space-valid regularization |
| Model | wide ResNet, 4-channel input → 1000 logits | a small, fast classifier on the compact input |
| Train | cross-entropy, OneCycle, 40 epochs | strong ImageNet accuracy, no pixels touched |

### The ideas to remember

1. **VAE latents are useful for recognition, not just generation.** The same frozen encoder that powers latent *diffusion* (nb 30) provides a compact, pretrained representation you can *classify* on. Good generative representations are broadly useful.
2. **This is foundation-model transfer.** Freeze a big pretrained model, extract its embedding once, train a small head on the embeddings. The VAE = the frozen foundation model; the `.npy` files = precomputed embeddings; the ResNet = the downstream head. Directly transferable to climate/weather foundation models.
3. **Per-file `.npy` vs memmap.** Store embeddings per-example in a class-folder tree when you need labels and folder semantics; use a single memmap (nb 30) for one big unlabeled stream. Different storage patterns for different downstream needs.
4. **Standardize per channel** (`xmean`/`xstd`) — a proper per-channel normalization, not a single scalar — and keep `denorm` to decode. The VAE's latent channels are not zero-centered by default.
5. **Augment in latent space carefully.** Translation (`pad+crop`) and erasing respect the latent's approximate spatial structure; flips/rotations/color jitter generally do not. Feature-space augmentation is powerful but must preserve meaning.

### Where this goes (end of the course)

This is the final course notebook. Together, notebooks 27–31 assembled the full modern generative-and-representation toolkit:

- **27** attention → **28** attention-conditioned diffusion U-Net → **29** VAE → **30** latent diffusion (generation) → **31** latent-space classification (recognition).

The through-line for your goals: a **pretrained encoder's latent space is a general-purpose substrate** — you can *generate* in it (diffusion) or *recognize* in it (classification), cheaply, because the expensive perceptual work is precomputed once. That is the foundation-model paradigm in miniature, and the exact pattern to carry into climate: pretrain (or adopt) an encoder for atmospheric/EO data, then build generative emulators *and* discriminative predictors on its embeddings.

### Suggested next steps

1. Re-read the three deep dives (latent-space classification, latent normalization, latent-space augmentation) — the transfer-learning framing is the transferable idea for your research.
2. Sketch the climate analog explicitly: what plays the role of the VAE (a pretrained ERA5/EO encoder), the `.npy` embeddings, and the downstream head, for one of your tasks (e.g. σ_LST regression or urban land-cover classification).
3. When satisfied, run `concept-extraction` (candidates: discriminative use of VAE latents, foundation-model transfer pattern, per-`.npy` label-preserving storage, per-channel latent normalization + denorm, latent-space augmentation validity).
4. Optionally `/colab` for a GPU-ready version (note: full ImageNet encode + 40-epoch train is very heavy — worth flagging) and `/html` to publish this final page.

**Phase B complete** — with this notebook, all of 27–31 are annotated. 🎢

---
